# Demo Notebook
This notebook shows how to train and evaluate a prediction model on a sample dataset. For full training runs, including HPO, please refer to the training scripts provided alongside the repository.

In [1]:
import sys
sys.path.append('../tcrsat')

In [2]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import CometLogger
import torch

from data import TCRSatDataset
from models.predictor import Predictor, EsmFreezeUnfreeze

from utils import CustomModelCheckpointCallback
from pytorch_lightning.callbacks import EarlyStopping

from pytorch_lightning.tuner import Tuner

/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/torch/cuda/__init__.py:68: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter an

In [3]:
path_sample_data = './sample/data.csv'

## Configuration
This only provides an examplatory configuration for this demo without considering performance. Please refer to the HPO script provided with this repository.

In [4]:
config = {
    'pretrained': 'facebook/esm2_t6_8M_UR50D',
    'head_config': {
        'activation': 'relu',
        'batch_norm': True,
        'dropout': 0.25,
        'hidden_neurons': 128,
        'num_hidden_layers': 2,
        'pool_type': 'start_token',
        'pool_num_heads': 4,
        'embedding_dim': None,
    },
    'lora_config': {
        'peft_type': None,
        'base_model_name_or_path': None,
        'task_type': None,
        'inference_mode': False,
        'r': 2**5,
        'target_modules': None,
        'lora_alpha': 2**5,
        'lora_dropout': 0.1,
        'fan_in_fan_out': False,
        'bias': 'none',
        'modules_to_save': None,
        'init_lora_weights': True,
    },
    'train_config': {
        'batch_size': 16,
        'lr': 10**(-5),
        'max_epochs': 10,
        'early_stopping': 3,
        'ignore_first': 0,
        'unfreeze_at_epoch': 0,
    },
    'data_config': {
        'path': path_sample_data,
        'sample_size': None,
        'split_col': 'group_stratified_split_0',
        'data_col': 'full',
        'max_seq_len': 463,
    }
}

## Initializing Model, Dataset, and Callbacks
Based on the configuration, we initialize the model and the dataset.

In [5]:
model = Predictor(**config)

/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/esm2_t6_8M_UR50D were not used when initializing EsmModel: ['lm_head.dense.bias', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSeque

Using LoRA
trainable params: 491520 || all params: 8331641 || trainable%: 5.899438057880794


In [6]:
train_dataset = TCRSatDataset(config=config['data_config'], split='train', tokenizer=model.tokenizer)
model.set_train_dataset(train_dataset)
val_dataset = TCRSatDataset(config=config['data_config'], split='val', tokenizer=model.tokenizer)
model.set_val_dataset(val_dataset)

In [7]:
callbacks = []
callbacks.append(CustomModelCheckpointCallback(ignore_first=config['train_config']['ignore_first'],
                                               dirpath=f'sample/test', monitor='val_auc', mode='max',
                                               filename='auc', save_last=True, save_top_k=1, verbose=False))
callbacks[0].CHECKPOINT_NAME_LAST = 'last-{epoch}'

callbacks.append(EarlyStopping(monitor='val_auc', patience=config['train_config']['early_stopping'],
                               mode='max', verbose=True))
callbacks.append(EsmFreezeUnfreeze(unfreeze_at_epoch=config['train_config']['unfreeze_at_epoch']))

In [8]:
logger = comet_logger = CometLogger(
    project_name="demo",
    save_dir="./sample",
)

CometLogger will be initialized in offline mode


## Train the model
We train the model for a limited amount of epochs. Following, we load the best model based on validation set AUC.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = pl.Trainer(accelerator=device,
                     max_epochs=config['train_config']['max_epochs'], log_every_n_steps=1, logger=logger,
                     callbacks=callbacks, num_sanity_val_steps=0, max_time='01:23:50:00')
tuner = Tuner(pl.Trainer(accelerator='cuda' if torch.cuda.is_available() else 'cpu', num_sanity_val_steps=0,
            callbacks=[EsmFreezeUnfreeze(unfreeze_at_epoch=-1)]))

model.batch_size = 16
trainer.fit(model)

/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /lustre/groups/imm01/workspace/felix.drost/miniforge ...
  rank_zero_warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:67: UserWarning: Starting from v1.9.0, `tensorboardX` has been removed as a dependenc

Model frozen


/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return [pytree], LeafSpec()
/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 224 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(
/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning: The dataloader, val_dataloader, does not have ma

Training: 0it [00:00, ?it/s]Unfreezing model at epoch  0
Epoch 0:   0%|          | 0/6 [00:00<?, ?it/s] 

/lustre/groups/imm01/workspace/felix.drost/miniforge/envs/tcrPredictionTest/lib/python3.10/site-packages/pytorch_lightning/callbacks/finetuning.py:221: UserWarning: The provided params to be frozen already exist within another group of this optimizer. Those parameters will be skipped.
HINT: Did you init your optimizer in `configure_optimizer` as such:
 <class 'torch.optim.adam.Adam'>(filter(lambda p: p.requires_grad, self.parameters()), ...) 
  rank_zero_warn(


Epoch 0: 100%|██████████| 6/6 [00:00<00:00,  9.21it/s, v_num=fd7d]
Validation: 0it [00:00, ?it/s]
Validation DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 86.53it/s]

/ictstr01/groups/imm01/workspace/felix.drost/26_natureSaturation/TCRprediction/analysis/../tcrsat/models/predictor.py:279: RuntimeWarning: invalid value encountered in divide
  optimal_threshold_prc = thresholds[np.nanargmax(2 * precision * recall / (precision + recall))]
COMET WARNING: Failing to save the matplotlib figure, reason: The figure is empty, please call log_figure() before calling show().


In [ ]:
model = Predictor.load_from_checkpoint('./sample/test/auc.ckpt', map_location=torch.device(device))

## Evaluate Model
We apply the best model to the sample dataset to obtain the prediction scores and calculate classification metrics on it.

In [ ]:
from collections import defaultdict
from tqdm import tqdm
import sklearn.metrics as metrics
import numpy as np
import seaborn as sb
import pandas as pd

In [ ]:
all_metric_results = []
seq_data = pd.read_csv(path_sample_data, index_col=0)

model.eval()

data_config = model.data_config.copy()
val_dataset = TCRSatDataset(config=data_config, split=None, tokenizer=model.tokenizer, data=seq_data)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False)

outputs = defaultdict(list)
with torch.no_grad():
    for batch in tqdm(val_dataloader):
        batch = [item.to(device) for item in batch]
        x, attention_mask, y = batch
        y_hat_logits = model(x, attention_mask)
        y_hat = model.transform_predict(y_hat_logits)

        outputs['y_hat'].append(y_hat.detach())
        outputs['y'].append(y.detach())

    y_hat = torch.cat(outputs['y_hat'])
    y = torch.cat(outputs['y'])

    y_hat = y_hat.cpu()
    y = y.cpu()
    seq_data[f'prediction'] = y_hat

    split_mask = seq_data[data_config['split_col']].to_numpy()
    metric_results = {}

    for s in ['train', 'val', 'test']:
        mask = (split_mask == s)
        fpr, tpr, thresholds = metrics.roc_curve(y[mask], y_hat[mask])
        optimal_threshold_auc = thresholds[np.argmax(tpr - fpr)]
        precision, recall, thresholds = metrics.precision_recall_curve(y[mask], y_hat[mask])
        optimal_threshold_prc = thresholds[np.nanargmax(2 * precision * recall / (precision + recall))]

        for metric_name, metric in model.evaluation.items():
            metric = metric.to('cpu')
            metric_results[f'{s}_{metric_name}'] = metric(y_hat[mask], y.int()[mask]).float().item()
    all_metric_results.append(metric_results)
result = pd.DataFrame(all_metric_results)

In [ ]:
result[['train_auc', 'val_auc', 'test_auc']]